# 04 · Baseline refusal and model choice

Measures each candidate model's refusal rate on harmful prompts and over-refusal on safe prompts (ADR-0002), checks fp16 is numerically safe on this GPU, and tunes the refusal / compliance first-token sets used by the cheap logit-based refusal score.

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · output of **01** |
| **Reads** | `eval_harm.jsonl`, `eval_overrefusal.jsonl` |
| **Writes** (`/kaggle/working/`) | `baseline/baseline_results.json`, `baseline/token_sets.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [ ]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

In [ ]:
from collections import Counter
import torch

# Tiny Aya (Cohere Labs, hackathon partner; gated: accept the licence on its Hub page) vs. the Qwen fallback.
CANDIDATES = ["CohereLabs/tiny-aya-global", os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)]
# Add more to compare (ADR-0002), e.g. "meta-llama/Llama-3.2-1B-Instruct" (gated), "google/gemma-2-2b-it".
N = 50 if not sp.SMOKE else 4
MAX_NEW = 128
OUT = sp.out_dir("baseline")
sp.hf_login()
harm = [x for x in sp.load_jsonl(sp.find_input("eval_harm.jsonl")) if x["split"] == "dev"][:N]
safe = [x for x in sp.load_jsonl(sp.find_input("eval_overrefusal.jsonl")) if x["split"] == "dev"][:N]
len(harm), len(safe)

In [ ]:
def redact(text, n_words=15):
    words = text.split()
    return " ".join(words[:n_words]) + (" [REDACTED]" if len(words) > n_words else "")


def tune_token_sets(tok, refused_texts, complied_texts, cover=0.9, k_max=8):
    """Most common first tokens of real refusals vs. real compliances, covering `cover` of each."""
    def top(texts):
        c = Counter(tok.encode(t.strip(), add_special_tokens=False)[:1][0] for t in texts if t.strip())
        ids, total, acc = [], sum(c.values()), 0
        for i, n in c.most_common(k_max):
            ids.append(i)
            acc += n
            if total and acc / total >= cover:
                break
        return ids
    r, c = top(refused_texts), top(complied_texts)
    overlap = set(r) & set(c)
    d = sp.default_token_sets(tok)
    return {"refuse": sorted(set(r) - overlap) or d["refuse"], "comply": sorted(set(c) - overlap) or d["comply"],
            "dropped_overlap": sorted(overlap)}

In [ ]:
results, token_sets = {}, {}
for name in CANDIDATES:
    model, tok = sp.load_model(name)
    logits = sp.last_logits(model, tok, [harm[0]["prompt"]])
    finite = bool(torch.isfinite(logits).all())
    print(name, "dtype", next(model.parameters()).dtype, "finite logits:", finite)
    if not finite:
        print("!! non-finite logits: reload with dtype=torch.float32")
        del model; sp.free()
        model, tok = sp.load_model(name, dtype=torch.float32)

    with sp.gpu_timer() as t:
        r_harm = sp.generate(model, tok, [x["prompt"] for x in harm], MAX_NEW, desc="harmful")
        r_safe = sp.generate(model, tok, [x["prompt"] for x in safe], MAX_NEW, desc="safe")
    refused = [r for r in r_harm if sp.is_refusal(r)]
    complied = [r for r in r_safe if not sp.is_refusal(r)]
    ts = tune_token_sets(tok, refused, complied)
    ld_h = sp.refusal_logit_diff(model, tok, [x["prompt"] for x in harm], ts)
    ld_s = sp.refusal_logit_diff(model, tok, [x["prompt"] for x in safe], ts)
    separation = float((ld_h[:, None] > ld_s[None, :]).float().mean())   # AUROC of the logit score
    results[name] = {
        "refusal_rate_harmful": sp.refusal_rate(r_harm),
        "over_refusal_safe": sp.refusal_rate(r_safe),
        "logit_diff_mean": {"harmful": float(ld_h.mean()), "safe": float(ld_s.mean())},
        "logit_auroc": separation,
        "gpu_seconds_generation": t["seconds"],
        "dtype": str(next(model.parameters()).dtype),
        "examples_harmful": [redact(r) for r in r_harm[:5]],
        "examples_safe": [redact(r, 30) for r in r_safe[:5]],
    }
    token_sets[name] = {**ts, "refuse_tokens": [tok.decode([i]) for i in ts["refuse"]],
                        "comply_tokens": [tok.decode([i]) for i in ts["comply"]]}
    print({k: v for k, v in results[name].items() if not k.startswith("examples")})
    print("token sets:", token_sets[name]["refuse_tokens"], "|", token_sets[name]["comply_tokens"])
    del model; sp.free()

In [ ]:
sp.save_json({"n": N, "max_new_tokens": MAX_NEW, "results": results, "provenance": sp.provenance()},
             OUT / "baseline_results.json")
sp.save_json(token_sets, OUT / "token_sets.json")
import pandas as pd
pd.DataFrame({k: {m: v for m, v in r.items() if not m.startswith("examples")} for k, r in results.items()}).T

## Decide (ADR-0002)

Pick the model with **high refusal on harmful prompts, low over-refusal, logit AUROC close to 1**, and that
all toolkits support (from 00). Copy the table above into `docs/adr/0002-model-selection.md`.
Notebooks 05–10 use `sp.DEFAULT_MODEL`: once a model is chosen, set it in `sp_utils.py` and upload a new
version of the `safetypulse-lib` dataset.

Check the token sets printed above: the refusal set should be tokens like "I"/"Sorry" and the compliance set
tokens like "Sure"/"Here"/"To". If AUROC < 0.9, the logit score is a poor stand-in for real refusals on this
model: look at the examples and adjust before going on.